In [1]:
# 1. AMBIENTE: VS Code / Jupyter Local
# Centralização de dependências. Importações internas removidas das funções 
# para garantir escopo global e imediata detecção de ausência de bibliotecas.
# ==============================================================================

# !pip install -U "google-genai"
# !pip install python-dotenv networkx matplotlib squarify plotly kaleido
import os
import re
import time
import json
import random
import textwrap
from dotenv import load_dotenv
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
from google import genai
from google.genai import types
from google.genai import errors
import plotly.express as px
import plotly.graph_objects as go
import squarify
import sys
import time
import typing_extensions as typing

In [2]:
# 2. Schemas rígidos para forçar a LLM a retornar JSON perfeitamente formatado.
# Nomenclaturas alinhadas hierarquicamente (Código -> Subtema -> Tema).
# ==============================================================================

class CodigoGerado(typing.TypedDict):
    codigo: str
    justificativa: str
    trecho_original: str
    definicao_operacional_curta: str

class DescricaoMestre(typing.TypedDict):
    descricao: str
    elementos_fagocitados: list[str]

class MapeamentoSubtema(typing.TypedDict):
    codigo: str
    subtema_alocado: str
    justificativa_alocacao: str        # NOVO: Fundamentação científica da fusão hierárquica
    definicao_operacional_curta: str   # NOVO: Âncora de controle conceitual (1 a 2 frases) 

class MapeamentoTema(typing.TypedDict):
    subtema: str
    tema_alocado: str
    justificativa_alocacao: str        # NOVO: Justificativa da convergência
    definicao_operacional_curta: str   # NOVO: Âncora de barreira macro (1-2 frases)

In [ ]:
# 3. Configuração de diretórios e definição dos arquivos JSON como 'Fonte Única da Verdade'.
# O estado da aplicação é lido do disco no início de cada arquivo de entrevista e 
# sobrescrito no disco ao final, permitindo injeção de edições manuais do pesquisador.
# ==============================================================================
DIRETORIO_ENTREVISTAS = r"C:\Users\F3807943\Downloads\Análise Temática\Entrevistas"
MODELO_LLM = "gemini-2.5-pro"
CHUNK_SIZE = 1500
CHUNK_OVERLAP = 300
MAX_TENTATIVAS = 15
DELAY_API = 5

DIRETORIO_CHECKPOINTS = os.path.join(DIRETORIO_ENTREVISTAS, "Checkpoints")
DIRETORIO_CACHE_F1 = os.path.join(DIRETORIO_CHECKPOINTS, "Fase1_Chunks")
DIRETORIO_GRAFICOS = os.path.join(DIRETORIO_ENTREVISTAS, "Graficos")

os.makedirs(DIRETORIO_CHECKPOINTS, exist_ok=True)
os.makedirs(DIRETORIO_CACHE_F1, exist_ok=True)
os.makedirs(DIRETORIO_GRAFICOS, exist_ok=True)

ARQUIVO_LOG_PROMPTS = os.path.join(DIRETORIO_CHECKPOINTS, "auditoria_prompts.md")

# Arquivos Mestre JSON (Fontes da Verdade Editáveis)
ARQUIVO_CODEBOOK_CODIGOS = os.path.join(DIRETORIO_CHECKPOINTS, "1_codebook_codigos.json")
ARQUIVO_CODEBOOK_SUBTEMAS = os.path.join(DIRETORIO_CHECKPOINTS, "2_codebook_subtemas.json")
ARQUIVO_CODEBOOK_TEMAS = os.path.join(DIRETORIO_CHECKPOINTS, "3_codebook_temas.json")
ARQUIVO_RASTREABILIDADE = os.path.join(DIRETORIO_CHECKPOINTS, "0_rastreabilidade_base.json")
ARQUIVO_HISTORICO_FUSOES = os.path.join(DIRETORIO_CHECKPOINTS, "historico_fagocitose.json")
ARQUIVO_LOG_CONSOLE = os.path.join(DIRETORIO_CHECKPOINTS, "auditoria_console.log")

load_dotenv()

# Instâncias Dinâmicas em Memória
Codebook_Codigos = {}
Codebook_Subtemas = {}
Codebook_Temas = {}
Rastreabilidade_Base = []
Metricas_Saturacao = []

api_key = os.environ.get("GEMINI_API_KEY")
if not api_key:
    raise ValueError("Variável de ambiente GEMINI_API_KEY não localizada.")
client = genai.Client(api_key=api_key)

In [4]:
# 4. CONTROLE DE ESTADO E INVOCAÇÃO LLM (AUDITORIA AVANÇADA)
# ==============================================================================

def ordenar_arquivos_cronologicamente(diretorio: str) -> list[str]:
    arquivos = [f for f in os.listdir(diretorio) if f.endswith(".txt")]
    def extrair_ordem(nome: str) -> int:
        match = re.search(r"Entrevistado A(\d+)", nome)
        return int(match.group(1)) if match else 9999
    return sorted(arquivos, key=extrair_ordem)

def segmentar_texto(texto: str, chunk_size: int, overlap: int) -> list[str]:
    palavras = texto.split()
    chunks = []
    passo = chunk_size - overlap
    for i in range(0, len(palavras), passo):
        chunk = palavras[i : i + chunk_size]
        chunks.append(" ".join(chunk))
        if i + chunk_size >= len(palavras):
            break
    return chunks

def invocar_llm(prompt: str, schema: typing.Any, instrucao_sistema: str, log_contexto: str = "PROCESSO GLOBAL") -> dict | list:
    max_tentativas = MAX_TENTATIVAS
    espera_inicial = DELAY_API
    
    for tentativa in range(1, max_tentativas + 1):
        try:
            # Aplica o tempo de espera calculado para a rodada atual
            if tentativa > 1:
                tempo_recuo = espera_inicial * (2 ** (tentativa - 2))
                # Limita o tempo máximo de espera a 120 segundos para evitar travamentos infinitos
                tempo_recuo = min(tempo_recuo, 120) 
                print(f"  [ALERTA API] Aguardando {tempo_recuo}s antes da tentativa {tentativa}/{max_tentativas}...")
                time.sleep(tempo_recuo)
            else:
                time.sleep(espera_inicial)
                
            resposta = client.models.generate_content(
                model=MODELO_LLM,
                contents=prompt,
                config=types.GenerateContentConfig(
                    system_instruction=instrucao_sistema,
                    temperature=0.0,
                    response_mime_type="application/json",
                    response_schema=schema,
                )
            )
            
            # Se a execução obteve sucesso, grava os metadados no log de auditoria
            with open(ARQUIVO_LOG_PROMPTS, "a", encoding="utf-8") as f:
                f.write(f"\n{'#'*80}\n")
                f.write(f"CONTEXTO DE EXECUÇÃO: {log_contexto} | DATA/HORA: {time.strftime('%Y-%m-%d %H:%M:%S')}\n")
                f.write(f"{'#'*80}\n")
                f.write(f"[INSTRUÇÃO DE SISTEMA]\n{instrucao_sistema}\n\n")
                f.write(f"[-] PROMPT ENVIADO:\n{prompt}\n\n")
                f.write(f"[+] RETORNO LLM (JSON):\n{resposta.text.strip()}\n\n")
                
            return json.loads(resposta.text.strip())
            
        except errors.APIError as e:
            # Erros de autenticação ou validação de parâmetros devem quebrar o código imediatamente
            if e.code in [400, 401, 403, 422]:
                raise e
                
            print(f"\n[ALERTA API] Instabilidade detectada (Erro {e.code}) no contexto: {log_contexto}.")
            print(f"  -> Falha na tentativa {tentativa} de {max_tentativas}.")
            
            # Se atingir o teto de 15 tentativas sem sucesso, força a quebra real do pipeline
            if tentativa == max_tentativas:
                print(f"\n[ERRO CRÍTICO] Limite exaustivo de {max_tentativas} tentativas atingido sem resposta da API.")
                raise e

def carregar_estado_json():
    global Codebook_Codigos, Codebook_Subtemas, Codebook_Temas, Rastreabilidade_Base
    if os.path.exists(ARQUIVO_CODEBOOK_CODIGOS):
        with open(ARQUIVO_CODEBOOK_CODIGOS, 'r', encoding='utf-8') as f: Codebook_Codigos.clear(); Codebook_Codigos.update(json.load(f))
    if os.path.exists(ARQUIVO_CODEBOOK_SUBTEMAS):
        with open(ARQUIVO_CODEBOOK_SUBTEMAS, 'r', encoding='utf-8') as f: Codebook_Subtemas.clear(); Codebook_Subtemas.update(json.load(f))
    if os.path.exists(ARQUIVO_CODEBOOK_TEMAS):
        with open(ARQUIVO_CODEBOOK_TEMAS, 'r', encoding='utf-8') as f: Codebook_Temas.clear(); Codebook_Temas.update(json.load(f))
    if os.path.exists(ARQUIVO_RASTREABILIDADE):
        with open(ARQUIVO_RASTREABILIDADE, 'r', encoding='utf-8') as f: Rastreabilidade_Base.clear(); Rastreabilidade_Base.extend(json.load(f))

def salvar_estado_json():
    with open(ARQUIVO_CODEBOOK_CODIGOS, 'w', encoding='utf-8') as f: json.dump(Codebook_Codigos, f, ensure_ascii=False, indent=4)
    with open(ARQUIVO_CODEBOOK_SUBTEMAS, 'w', encoding='utf-8') as f: json.dump(Codebook_Subtemas, f, ensure_ascii=False, indent=4)
    with open(ARQUIVO_CODEBOOK_TEMAS, 'w', encoding='utf-8') as f: json.dump(Codebook_Temas, f, ensure_ascii=False, indent=4)
    with open(ARQUIVO_RASTREABILIDADE, 'w', encoding='utf-8') as f: json.dump(Rastreabilidade_Base, f, ensure_ascii=False, indent=4)

In [ ]:
# 5. MOTOR METODOLÓGICO: EXTRAÇÃO E ABSTRAÇÃO ESTRUTURADA
# Prompts mantidos estritamente conforme exigido. A lógica injeta dinamicamente o  
# estado existente vs. órfãos na formatação de contexto para propiciar avanço 
# incremental, atuando exclusivamente em objetos sem processamento prévio.
# ==============================================================================

def executar_fase_1(arquivo: str):
    instrucao = (
        "Você é um Analista Qualitativo Sênior especializado executando a Fase 2 (Geração de subtemas Iniciais) da Análise Temática Reflexiva de Braun e Clarke. Você opera sob o paradigma interpretativo sênior.\n\n"
        "Suas atividades estão centradas em identificar elementos que auxiliem na análise do objetivo da pesquisa. Como componente de uma equipe científica, colabore para que as conclusões sejam imparciais, porém focadas no objetivo. Subtemas e temas distantes do Objetivo de Pesquisa devem ser sumariamente ignorados.\n"
        "Objetivo da Pesquisa: Como Automação Inteligente pode ser efetivamente implementada para aumentar as capacidades cognitivas e contribuindo para o processo decisório e qual framework emerge desse processo de implementação.\n\n"
        "DEFINIÇÕES CONCEITUAIS (BRAUN & CLARKE):\n"
        "- CÓDIGO: Nível granular e fundamental. Ferramenta analítica básica que captura uma única observação ou faceta da informação (obviedade ou significado latente). Não é um conceito amplo, mas uma etiqueta específica para um trecho de dados.\n"
        "- SUBTEMA: Nível estrutural subordinado. Aplicado estritamente para organizar temas amplos ou complexos. Demonstra a hierarquia da informação, estruturando nuances, dimensões ou categorias secundárias do conceito central pai.\n"
        "- TEMA: Alto nível de abstração. Padrão de significado compartilhado em todo o conjunto de dados, unido por um conceito central organizador. Atua como um cristal multifacetado que reúne diversos códigos para contar uma história interpretativa sobre os dados em relação à pergunta de pesquisa. Proibido criar temas que sejam meros resumos de tópicos ou categorias descritivas de uma palavra (ex: 'Vantagens' ou 'Barreiras').\n\n"
        "DIRETRIZES PARA REDAÇÃO DAS DEFINIÇÕES:\n"
        "Para cada elemento gerado (código, subtema ou tema), forneça uma definição rigorosa e analítica formatada como uma descrição narrativa concisa (2 a 4 frases). A definição deve obrigatoriamente:\n"
        "1. Essência: Explicar a ideia central e o conteúdo do padrão identificado.\n"
        "2. Escopo: Demarcar claramente as fronteiras do elemento, especificando o que ele abrange e o que não abrange, para eliminar sobreposição com outros elementos.\n"
        "3. Relevância: Explicar a importância conceitual do elemento, indicando por que ele responde à pergunta de pesquisa e como se conecta à narrativa geral dos dados.\n\n"
        "DIRETRIZES ABSOLUTAS:\n"
        "1. Alvo Analítico: Codifique EXCLUSIVAMENTE as falas do entrevistado. Ignore as falas dos entrevistadores para extração, usando-as apenas como âncora de contexto.\n"
        "2. Fidelidade Empírica: Extraia a citação direta literal e exata da fala, sem parafrasear, sem corrigir gramática e sem omitir termos.\n"
        "3. Nomenclatura: O nome do código deve ser INCISIVO, CURTO E DIRETO.\n"
        "4. Profundidade: Reflita constructos conceituais de significado latente adequados para um artigo de alto impacto (Q1).\n"
        "5. Integridade Linguística: Preserve expressões idiomáticas, gírias ou termos técnicos específicos usados pelo entrevistado. Mantenha erros gramaticais ou expressões coloquiais originais para capturar integralmente o contexto cultural, social e as nuances de significado latente."
    )
    
    caminho_arquivo = os.path.join(DIRETORIO_ENTREVISTAS, arquivo)
    arquivo_checkpoint = os.path.join(DIRETORIO_CACHE_F1, f"cache_{arquivo}.json")
    
    dados_salvos_arquivo = []
    chunks_processados = set()
    if os.path.exists(arquivo_checkpoint):
        with open(arquivo_checkpoint, 'r', encoding='utf-8') as f:
            dados_salvos_arquivo = json.load(f)
            chunks_processados = {item["ID_Chunk"] for item in dados_salvos_arquivo}
    
    with open(caminho_arquivo, 'r', encoding='utf-8') as f:
        texto = f.read()
        
    chunks = segmentar_texto(texto, CHUNK_SIZE, CHUNK_OVERLAP)

    for id_chunk, chunk in enumerate(chunks):
        if id_chunk in chunks_processados:
            continue 
            
        # --- NOVO BLOCO: CONSTRUÇÃO DO DICIONÁRIO CONCEITUAL COMPACTO ---
        # Em vez de passar apenas chaves, mapeia o nome do código para sua definição operacional curta
        codebook_contexto = {}
        for k, v in Codebook_Codigos.items():
            codebook_contexto[k] = v.get("definicao_operacional_curta", "Sem definição prévia.")
        # ----------------------------------------------------------------
        prompt = (
            f"Analise o segmento de texto abaixo.\n"
            f"TEXTO (Chunk {id_chunk}):\n<chunk>\n{chunk}\n</chunk>\n\n"
            f"CODEBOOK ATUAL (Estrutura conceitual existente): {json.dumps(codebook_contexto, ensure_ascii=False)}\n\n"
            "AÇÃO EXIGIDA:\n"
            "Realize uma varredura exaustiva. Extraia até 5 unidades de significado presentes no chunk, priorizando os mais distantes semanticamente e que cumpram o objetivo da pesquisa.\n\n"
            "DIRETRIZ DE PREVENÇÃO À FRAGMENTAÇÃO (FAGOCITAÇÃO OPERACIONAL):\n"
            "Antes de criar um novo código, avalie rigorosamente o CODEBOOK ATUAL. É PROIBIDO criar códigos que sejam sinônimos, variações gramaticais ou sobreposições conceituais de itens já existentes.\n"
            "Se o trecho analisado se alinhar à 'definicao_operacional_curta' de um código existente, REUTILIZE o nome exato da chave e adapte a justificativa ao novo contexto.\n"
            "Apenas se o conceito for categoricamente inédito, crie um novo nome e forneça uma 'definicao_operacional_curta' inédita (estritamente limitada a 1 ou 2 frases curtas).\n\n"
            "Retorne os dados formatados estritamente conforme o schema JSON solicitado pelo sistema."
        )          
        
        ctx_log = f"FASE 1 | Arquivo: {arquivo} | Chunk: {id_chunk}"
        resultado = invocar_llm(prompt, list[CodigoGerado], instrucao, log_contexto=ctx_log)
        
        for item in resultado:
            cod_nome = item['codigo'].upper().strip()
            item_rastreio = {
                "Arquivo_Origem": arquivo,
                "ID_Chunk": id_chunk,
                "Trecho_Original": item['trecho_original'],
                "Codigo_Atribuido": cod_nome,
                "Justificativa_Original": item['justificativa']
            }
            dados_salvos_arquivo.append(item_rastreio)
            Rastreabilidade_Base.append(item_rastreio)
            
            # ATUALIZAÇÃO DA ESTRUTURA: Incorpora a definição curta ao dicionário em memória
            if cod_nome not in Codebook_Codigos:
                Codebook_Codigos[cod_nome] = {
                    "descricao": "", 
                    "definicao_operacional_curta": item.get('definicao_operacional_curta', '').strip(),
                    "evidencias_brutas": []
                }
            elif item.get('definicao_operacional_curta') and not Codebook_Codigos[cod_nome]["definicao_operacional_curta"]:
                Codebook_Codigos[cod_nome]["definicao_operacional_curta"] = item['definicao_operacional_curta'].strip()
            
            Codebook_Codigos[cod_nome]["evidencias_brutas"].append({
                "citacao": item['trecho_original'],
                "justificativa": item['justificativa']
            })
        
        with open(arquivo_checkpoint, 'w', encoding='utf-8') as f:
            json.dump(dados_salvos_arquivo, f, ensure_ascii=False, indent=4)
            
        print(f"  [FASE 1] Chunk {id_chunk+1}/{len(chunks)} processado. | {len(resultado)} extrações.")

def executar_fase_intermediaria_1():
    instrucao = (
        "Você é um Analista Qualitativo Sênior. "
        "Suas atividades está centrada em um foco o foco máximo que é o de responder ou identificar elementos que nos ajude a analisar o objetivo da pesquisa. Pois, como componente de uma equipe de pesquisa científica deve colaborar para que suas conclusões sejam imparciais porém focados em atingir o obejtivo de pesquisa. Isto é, subtemas e temas que estejam distântes do Objetivo de Pesquisa deve ser ignorado.\n"
        "Objetivo da Pesquisa: Como Automação Inteligente pode ser efetivamente implementada para aumentar as capacidades cognitivas e contribuindo para o processo decisório e qual framework emerge desse processo de implementação.\n\n"
        "DEFINIÇÕES CONCEITUAIS (BRAUN & CLARKE):\n"
        "- CÓDIGO: Nível granular e fundamental. Ferramenta analítica básica que captura uma única observação ou faceta da informação (obviedade ou significado latente). Não é um conceito amplo, mas uma etiqueta específica para um trecho de dados.\n"
        "- SUBTEMA: Nível estrutural subordinado. Aplicado estritamente para organizar temas amplos ou complexos. Demonstra a hierarquia da informação, estruturando nuances, dimensões ou categorias secundárias do conceito central pai.\n"
        "- TEMA: Alto nível de abstração. Padrão de significado compartilhado em todo o conjunto de dados, unido por um conceito central organizador. Atua como um cristal multifacetado que reúne diversos códigos para contar uma história interpretativa sobre os dados em relação à pergunta de pesquisa. Proibido criar temas que sejam meros resumos de tópicos ou categorias descritivas de uma palavra (ex: 'Vantagens' ou 'Barreiras').\n\n"
        "DIRETRIZES PARA REDAÇÃO DAS DEFINIÇÕES:\n"
        "Para cada elemento gerado (código, subtema ou tema), forneça uma definição rigorosa e analítica formatada como uma descrição narrativa concisa (2 a 4 frases). A definição deve obrigatoriamente:\n"
        "1. Essência: Explicar a ideia central e o conteúdo do padrão identificado.\n"
        "2. Escopo: Demarcar claramente as fronteiras do elemento, especificando o que ele abrange e o que não abrange, para eliminar sobreposição com outros elementos.\n"
        "3. Relevância: Explicar a importância conceitual do elemento, indicando por que ele responde à pergunta de pesquisa e como se conecta à narrativa geral dos dados.\n\n"
        "Sua tarefa é ler um conjunto de justificativas brutas extraídas do campo e sintetizá-las em uma Descrição Mestre única, analítica e conceitual para o código informado.\n\n"
        "Preserve expressões idiomáticas, gírias ou termos técnicos específicos usados pelo entrevistado. Mantenha erros gramaticais ou expressões coloquiais originais para capturar integralmente o contexto cultural, social e as nuances de significado latente."        
    )
    
    for codigo, dados in Codebook_Codigos.items():
        if dados.get("descricao"):
            continue 
            
        evs = dados.get("evidencias_brutas", [])
        # Deduplicação de dicionários para limpar repetições exatas
        evs_unicas = [dict(t) for t in {tuple(d.items()) for d in evs}]
        
        # EXAUSTÃO EMPÍRICA: Desativação do limitador para submeter 100% dos dados
        amostra = evs_unicas
        def_inicial = dados.get("definicao_operacional_curta", "Não registrada.")

        prompt = (
            f"NOME DO CÓDIGO: {codigo}\n"
            f"DEFINIÇÃO OPERACIONAL INICIAL (Âncora de Controle): {def_inicial}\n"
            f"DADOS EMPÍRICOS ACUMULADOS (Citações Diretas e Justificativas): {json.dumps(amostra, ensure_ascii=False)}\n\n"
            "AÇÃO EXIGIDA:\n"
            "Com base no pacote de dados brutos acima (nome, definição operacional inicial, citações diretas literais e justificativas de campo), consolide a Descrição Mestre definitiva do elemento.\n"
            "Aplique rigorosamente as DIRETRIZES PARA REDAÇÃO DAS DEFINIÇÕES presentes na instrução de sistema (Essência, Escopo e Relevância).\n\n"
            "DIRETRIZ DE FAGOCITOSE OPERACIONAL E SANEAMENTO DE SINÔNIMOS:\n"
            "- Analise criticamente se os dados empíricos acumulados revelam que este código gerou uma fragmentação excessiva, sobrepondo-se semanticamente a outros conceitos parecidos ou sinônimos do estudo.\n"
            "- Force a fusão conceitual: utilize a descrição mestre para expandir o significado deste código de modo que ele absorva e centralize as variações marginais de termos correlatos, unificando a categoria.\n"
            "- Use o critério de 'Escopo' para desenhar uma linha divisória clara, explicitando de forma cirúrgica o que este código abrange e o que ele NÃO abrange, blindando o codebook contra novas duplicidades nas próximas iterações.\n\n"
            "REGRA LINGUÍSTICA ABSOLUTA: Baseie-se fortemente nas falas reais para manter o dialeto, termos técnicos corporativos, expressões coloquiais e erros gramaticais originais do entrevistado, capturando o real significado latente do fenômeno corporativo investigado.\n\n"
            "Retorne a síntese formatada estritamente conforme o schema JSON de descrição."
        )
        
        ctx_log = f"SÍNTESE INT. 1 | Código: {codigo}"
        resultado = invocar_llm(prompt, DescricaoMestre, instrucao, log_contexto=ctx_log)
        Codebook_Codigos[codigo]["descricao"] = resultado["descricao"]
        print(f"  [SÍNTESE CÓDIGO] {codigo} -> Escrito.")

def executar_fase_2():
    instrucao = (
        "Você é um Analista Qualitativo Sênior executando a Etapa 2 da Fase 3 de Braun e Clarke. Você deve agrupar códigos em 'subtemas de Primeiro Nível' (Categorias).\n\n"
        "Suas atividades está centrada em um foco o foco máximo que é o de responder ou identificar elementos que nos ajude a analisar o objetivo da pesquisa. Pois, como componente de uma equipe de pesquisa científica deve colaborar para que suas conclusões sejam imparciais porém focados em atingir o obejtivo de pesquisa. Isto é, subtemas e temas que estejam distântes do Objetivo de Pesquisa deve ser ignorado.\n"
        "Objetivo da Pesquisa: Como Automação Inteligente pode ser efetivamente implementada para aumentar as capacidades cognitivas e contribuindo para o processo decisório e qual framework emerge desse processo de implementação.\n\n"
        "DEFINIÇÕES CONCEITUAIS (BRAUN & CLARKE):\n"
        "- CÓDIGO: Nível granular e fundamental. Ferramenta analítica básica que captura uma única observação ou faceta da informação (obviedade ou significado latente). Não é um conceito amplo, mas uma etiqueta específica para um trecho de dados.\n"
        "- SUBTEMA: Nível estrutural subordinado. Aplicado estritamente para organizar temas amplos ou complexos. Demonstra a hierarquia da informação, estruturando nuances, dimensões ou categorias secundárias do conceito central pai.\n"
        "- TEMA: Alto nível de abstração. Padrão de significado compartilhado em todo o conjunto de dados, unido por um conceito central organizador. Atua como um cristal multifacetado que reúne diversos códigos para contar uma história interpretativa sobre os dados em relação à pergunta de pesquisa. Proibido criar temas que sejam meros resumos de tópicos ou categorias descritivas de uma palavra (ex: 'Vantagens' ou 'Barreiras').\n\n"
        "DIRETRIZES PARA REDAÇÃO DAS DEFINIÇÕES:\n"
        "Para cada elemento gerado (código, subtema ou tema), forneça uma definição rigorosa e analítica formatada como uma descrição narrativa concisa (2 a 4 frases). A definição deve obrigatoriamente:\n"
        "1. Essência: Explicar a ideia central e o conteúdo do padrão identificado.\n"
        "2. Escopo: Demarcar claramente as fronteiras do elemento, especificando o que ele abrange e o que não abrange, para eliminar sobreposição com outros elementos.\n"
        "3. Relevância: Explicar a importância conceitual do elemento, indicando por que ele responde à pergunta de pesquisa e como se conecta à narrativa geral dos dados.\n\n"
        "REGRAS DE FORMATAÇÃO E AGRUPAMENTO:\n"
        "- A matriz resultante DEVE ser limitada a no MÁXIMO 15 subtemas lógicos e abrangentes no total de todo o projeto.\n\n"
        "- REGRA ABSOLUTA: O nome do subtema deve ter MÁXIMA SIMPLICIDADE (1 a 3 palavras no máximo).\n"
        "- Evite nomear subtemas com o mesmo nome de códigos existentes para evitar confusão hierárquica. Subtemas devem ser categorias organizadoras, não rótulos de observações específicas.\"\n"
        "- Integridade Linguística: Preserve expressões idiomáticas, gírias ou termos técnicos específicos usados pelo entrevistado. Mantenha erros gramaticais ou expressões coloquiais originais para capturar integralmente o contexto cultural, social e as nuances de significado latente.\n"
        "- subtemas devem ser expressos como FRASE CONCISA OU PALAVRA-CHAVE (Ex: 'Liderança', 'Rigidez Processual').\n"        
    )
       
    # Isola códigos órfãos e monta payload rico com contexto empírico
    codigos_alocados = set()
    for dados in Codebook_Subtemas.values(): codigos_alocados.update(dados.get("codigos_vinculados", []))
    
    codigos_orfaos = []
    for k, v in Codebook_Codigos.items():
        if k not in codigos_alocados:
            evs = v.get("evidencias_brutas", [])
            evs_unicas = [dict(t) for t in {tuple(d.items()) for d in evs}]
            
            # ESTABILIZAÇÃO: Fatiamento fixo das 5 primeiras citações em vez de amostragem aleatória
            amostra_estabilizada = evs_unicas[:5] 
            
            codigos_orfaos.append({
                "codigo": k, 
                "definicao_teorica": v.get("descricao", ""),
                "citacoes_empiricas": [e["citacao"] for e in amostra_estabilizada]
            })
    
    if not codigos_orfaos:
        print("  [FASE 2] Nenhum código órfão para mapeamento.")
        return

    # ANCORAGEM SEMÂNTICA: Extração dos subtemas existentes acompanhados de seus escopos teóricos
    subtemas_contexto = {}
    for k, v in Codebook_Subtemas.items():
        subtemas_contexto[k] = v.get("descricao", "Sem descrição definida até o momento.")

    prompt = (
        f"CONTEXTO GLOBAL - SUBTEMAS EXISTENTES E SEUS ESCOPOS: {json.dumps(subtemas_contexto, ensure_ascii=False)}\n\n"
        "Abaixo está a lista de novos códigos órfãos extraídos do campo. Para cada código, é fornecida sua 'definicao_teorica' (síntese teórica) e 'citacoes_empiricas' (falas literais).\n\n"
        "AÇÃO EXIGIDA:\n"
        "1. Analise os códigos órfãos e agrupe-os em Subtemas (Categorias estruturantes).\n\n"
        "DIRETRIZ DE ESTABILIZAÇÃO TAXONÔMICA (REPLICABILIDADE CIENTÍFICA):\n"
        "- É EXPRESSAMENTE PROIBIDO criar um novo Subtema se o conceito analisado puder ser acomodado, expandido ou contextualizado dentro do escopo de um dos SUBTEMAS EXISTENTES informados acima.\n"
        "- Evite criar novas categorias que representem sinônimos, sobreposições ou meras variações de nomenclatura de estruturas já estabelecidas.\n"
        "- Se um código órfão se alinhar conceitualmente a um SUBTEMA EXISTENTE, REUTILIZE o nome exato da chave de forma idêntica.\n"
        "- A criação de um novo subtema deve ser uma exceção absoluta, reservada exclusivamente para fenômenos empíricos categoricamente inéditos e impossíveis de serem integrados às estruturas vigentes.\n"
        "- Para cada alocação, preencha rigorosamente a 'justificativa_alocacao' fundamentando a conexão científica. Caso um subtema INÉDITO seja criado, forneça uma 'definicao_operacional_curta' de controle (máximo 2 frases).\n\n"
        f"LISTA DE CÓDIGOS PARA ANÁLISE:\n{json.dumps(codigos_orfaos, ensure_ascii=False)}\n\n"
        "Retorne o mapeamento relacional estritamente conforme o schema JSON solicitado."
    )    
      
    ctx_log = f"FASE 2 | Mapeamento de {len(codigos_orfaos)} códigos órfãos"
    resultado = invocar_llm(prompt, list[MapeamentoSubtema], instrucao, log_contexto=ctx_log)
    
    for item in resultado:
        cod_nome = item["codigo"].upper().strip()
        subtema_nome = item["subtema_alocado"].upper().strip()
        
        if subtema_nome not in Codebook_Subtemas:
            # Inicialização com os novos metadados conceituais integrados
            Codebook_Subtemas[subtema_nome] = {
                "descricao": "", 
                "definicao_operacional_curta": item.get("definicao_operacional_curta", "").strip(),
                "codigos_vinculados": [], 
                "justificativas_agrupamento": [],
                "precisa_revisao": True
            }
            
        if cod_nome not in Codebook_Subtemas[subtema_nome]["codigos_vinculados"]:
            Codebook_Subtemas[subtema_nome]["codigos_vinculados"].append(cod_nome)
            Codebook_Subtemas[subtema_nome]["justificativas_agrupamento"].append({
                "codigo": cod_nome,
                "justificativa": item.get("justificativa_alocacao", "")
            })
            Codebook_Subtemas[subtema_nome]["precisa_revisao"] = True
            
        print(f"  [MAP] {cod_nome} -> {subtema_nome}")

def executar_fase_intermediaria_2():
    instrucao = (
        "Você é um Analista Qualitativo Sênior. "
        "Suas atividades está centrada em um foco o foco máximo que é o de responder ou identificar elementos que nos ajude a analisar o objetivo da pesquisa. Pois, como componente de uma equipe de pesquisa científica deve colaborar para que suas conclusões sejam imparciais porém focados em atingir o obejtivo de pesquisa. Isto é, subtemas e temas que estejam distântes do Objetivo de Pesquisa deve ser ignorado.\n"
        "Objetivo da Pesquisa: Como Automação Inteligente pode ser efetivamente implementada para aumentar as capacidades cognitivas e contribuindo para o processo decisório e qual framework emerge desse processo de implementação.\n\n"
        "DEFINIÇÕES CONCEITUAIS (BRAUN & CLARKE):\n"
        "- CÓDIGO: Nível granular e fundamental. Ferramenta analítica básica que captura uma única observação ou faceta da informação (obviedade ou significado latente). Não é um conceito amplo, mas uma etiqueta específica para um trecho de dados.\n"
        "- SUBTEMA: Nível estrutural subordinado. Aplicado estritamente para organizar temas amplos ou complexos. Demonstra a hierarquia da informação, estruturando nuances, dimensões ou categorias secundárias do conceito central pai.\n"
        "- TEMA: Alto nível de abstração. Padrão de significado compartilhado em todo o conjunto de dados, unido por um conceito central organizador. Atua como um cristal multifacetado que reúne diversos códigos para contar uma história interpretativa sobre os dados em relação à pergunta de pesquisa. Proibido criar temas que sejam meros resumos de tópicos ou categorias descritivas de uma palavra (ex: 'Vantagens' ou 'Barreiras').\n\n"
        "DIRETRIZES PARA REDAÇÃO DAS DEFINIÇÕES:\n"
        "Para cada elemento gerado (código, subtema ou tema), forneça uma definição rigorosa e analítica formatada como uma descrição narrativa concisa (2 a 4 frases). A definição deve obrigatoriamente:\n"
        "1. Essência: Explicar a ideia central e o conteúdo do padrão identificado.\n"
        "2. Escopo: Demarcar claramente as fronteiras do elemento, especificando o que ele abrange e o que não abrange, para eliminar sobreposição com outros elementos.\n"
        "3. Relevância: Explicar a importância conceitual do elemento, indicando por que ele responde à pergunta de pesquisa e como se conecta à narrativa geral dos dados.\n\n"
        "Sua tarefa é gerar uma Descrição Mestre de alto nível científico (Q1) para uma Categoria (subtema), baseando-se nas evidências basais agregadas dos códigos que a compõem."
        "Integridade Linguística: Preserve expressões idiomáticas, gírias ou termos técnicos específicos usados pelo entrevistado. Mantenha erros gramaticais ou expressões coloquiais originais para capturar integralmente o contexto cultural, social e as nuances de significado latente."
    )

    for subtema, dados in Codebook_Subtemas.items():
        if dados.get("descricao") and not dados.get("precisa_revisao"):
            continue

        evidencias = []
        for cod_nome in dados["codigos_vinculados"]:
            if cod_nome in Codebook_Codigos:
                evidencias.extend(Codebook_Codigos[cod_nome].get("evidencias_brutas", []))
                
        evs_unicas = [dict(t) for t in {tuple(d.items()) for d in evidencias}]
        
        # EXAUSTÃO EMPÍRICA: Desativação do limitador randômico para processamento de 100% dos insumos
        amostra = evs_unicas
        
        descricao_anterior = dados.get("descricao", "")
        def_inicial = dados.get("definicao_operacional_curta", "Não registrada.")
        justificativas_historicas = dados.get("justificativas_agrupamento", [])
        ctx_log = f"SÍNTESE INT. 2 | Subtema: {subtema}"

        prompt = (
            f"NOME DA CATEGORIA (SUBTEMA): {subtema}\n"
            f"DEFINIÇÃO OPERACIONAL INICIAL: {def_inicial}\n"
            f"JUSTIFICATIVAS DE MAPEAMENTO DOS CÓDIGOS PAI: {json.dumps(justificativas_historicas, ensure_ascii=False)}\n"
            f"DESCRIÇÃO CONCEITUAL ACUMULADA: {descricao_anterior if descricao_anterior else 'Inexistente. Construção do escopo inicial.'}\n"
            f"EVIDÊNCIAS EMPÍRICAS INTEGRAIS (Citações e Justificativas de Códigos): {json.dumps(amostra, ensure_ascii=False)}\n\n"
            "AÇÃO EXIGIDA:\n"
            "Com base no pacote analítico completo acima, gere ou refine a Descrição Mestre definitiva deste Subtema (Categoria).\n"
            "Aplique rigorosamente as DIRETRIZES PARA REDAÇÃO DAS DEFINIÇÕES presentes na instrução de sistema (Essência, Escopo e Relevância).\n\n"
            "DIRETRIZ DE PRESERVAÇÃO DE NUANCES (EVITAR ACHATAMENTO TAXONÔMICO):\n"
            "- ATENÇÃO: É terminantemente proibido fundir ou absorver subtemas que representem facetas complementares, dimensões diferentes ou tensões distintas do mesmo fenômeno.\n"
            "- A fagocitose deve ser aplicada EXCLUSIVAMENTE para sinônimos perfeitos ou redundâncias de escrita flagrantes.\n"
            "- Utilize o critério de 'Escopo' para refinar as fronteiras lógicas desta categoria, garantindo que ela mantenha sua identidade analítica individualizada sem invadir ou engolir categorias vizinhas correlatas.\n\n"
            "REGRA LINGUÍSTICA ABSOLUTA: Baseie-se fortemente nas falas reais para manter o dialeto, termos corporativos, jargões e erros gramaticais originais do entrevistado, capturando o real significado latente do fenômeno corporativo investigado.\n\n"
            "Retorne a síntese formatada estritamente conforme o schema JSON de descrição."
        )
                                
        resultado = invocar_llm(prompt, DescricaoMestre, instrucao, log_contexto=ctx_log)
        Codebook_Subtemas[subtema]["descricao"] = resultado["descricao"]
        Codebook_Subtemas[subtema]["precisa_revisao"] = False
        
        # INTERCEPTAÇÃO DE FAGOCITOSE: Alinha a Matriz de Saturação limpando chaves sinônimas decretadas pela LLM
        lista_mortas = resultado.get("elementos_fagocitados", [])
        registrar_e_executar_fagocitose("SUBTEMA", subtema, lista_mortas)
        
        print(f"  [SÍNTESE SUBTEMA] {subtema} -> Atualizado/Escrito.")

def executar_fase_3():
    instrucao = (
        "Você é um Analista Qualitativo Sênior executando a Etapa 2 da Fase 3 de Braun e Clarke. Sua missão é agrupar as Categorias Intermediárias em Temas Estruturantes.\n\n"
        "Suas atividades está centrada em um foco o foco máximo que é o de responder ou identificar elementos que nos ajude a analisar o objetivo da pesquisa. Pois, como componente de uma equipe de pesquisa científica deve colaborar para que suas conclusões sejam imparciais porém focados em atingir o obejtivo de pesquisa. Isto é, subtemas e temas que estejam distântes do Objetivo de Pesquisa deve ser ignorado.\n"
        "Objetivo da Pesquisa: Como Automação Inteligente pode ser efetivamente implementada para aumentar as capacidades cognitivas e contribuindo para o processo decisório e qual framework emerge desse processo de implementação.\n\n"
        "DEFINIÇÕES CONCEITUAIS (BRAUN & CLARKE):\n"
        "- CÓDIGO: Nível granular e fundamental. Ferramenta analítica básica que captura uma única observação ou faceta da informação (obviedade ou significado latente). Não é um conceito amplo, mas uma etiqueta específica para um trecho de dados.\n"
        "- SUBTEMA: Nível estrutural subordinado. Aplicado estritamente para organizar temas amplos ou complexos. Demonstra a hierarquia da informação, estruturando nuances, dimensões ou categorias secundárias do conceito central pai.\n"
        "- TEMA: Alto nível de abstração. Padrão de significado compartilhado em todo o conjunto de dados, unido por um conceito central organizador. Atua como um cristal multifacetado que reúne diversos códigos para contar uma história interpretativa sobre os dados em relação à pergunta de pesquisa. Proibido criar temas que sejam meros resumos de tópicos ou categorias descritivas de uma palavra (ex: 'Vantagens' ou 'Barreiras').\n\n"
        "DIRETRIZES PARA REDAÇÃO DAS DEFINIÇÕES:\n"
        "Para cada elemento gerado (código, subtema ou tema), forneça uma definição rigorosa e analítica formatada como uma descrição narrativa concisa (2 a 4 frases). A definição deve obrigatoriamente:\n"
        "1. Essência: Explicar a ideia central e o conteúdo do padrão identificado.\n"
        "2. Escopo: Demarcar claramente as fronteiras do elemento, especificando o que ele abrange e o que não abrange, para eliminar sobreposição com outros elementos.\n"
        "3. Relevância: Explicar a importância conceitual do elemento, indicando por que ele responde à pergunta de pesquisa e como se conecta à narrativa geral dos dados.\n\n"
        "REGRAS DIRETIVAS:\n"
        "- A matriz final DEVE conter no MÁXIMO 10 Temas estruturantes.\n"
        "- Nomes de Temas devem ser frases descritivas curtas e abstratas (Ex: 'Desafios Estruturais do Trabalho Remoto').\n"
        "- REGRA ABSOLUTA: O nome do Tema deve ter MÁXIMA SIMPLICIDADE (1 a 3 palavras no máximo)."
        "- Integridade Linguística: Preserve expressões idiomáticas, gírias ou termos técnicos específicos usados pelo entrevistado. Mantenha erros gramaticais ou expressões coloquiais originais para capturar integralmente o contexto cultural, social e as nuances de significado latente."
        "- evite nomear temas com o mesmo nome de subtemas existentes para evitar confusão hierárquica. Temas devem ser categorias organizadoras de alto nível, não rótulos de observações específicas ou categorias intermediárias."
    )
    
    subtemas_alocados = set()
    for dados in Codebook_Temas.values(): subtemas_alocados.update(dados.get("subtemas_vinculados", []))
    
    subtemas_orfaos = []
    for k, v in Codebook_Subtemas.items():
        if k not in subtemas_alocados:
            evidencias = []
            for cod_nome in v.get("codigos_vinculados", []):
                if cod_nome in Codebook_Codigos:
                    evidencias.extend(Codebook_Codigos[cod_nome].get("evidencias_brutas", []))
            
            evs_unicas = [dict(t) for t in {tuple(d.items()) for d in evidencias}]
            # ESTABILIZAÇÃO: Fatiamento fixo das 10 primeiras ocorrências (Remove o random.sample)
            amostra_estabilizada = evs_unicas[:10] 
            
            subtemas_orfaos.append({
                "subtema": k, 
                "definicao_teorica": v.get("descricao", ""),
                "citacoes_empiricas": [e["citacao"] for e in amostra_estabilizada]
            })
    
    if not subtemas_orfaos:
        print("  [FASE 3] Nenhum subtema órfão para mapeamento.")
        return

    # ANCORAGEM SEMÂNTICA: Mapeia Temas existentes para suas definições conceituais (Remove a lista de strings plana)
    temas_contexto = {}
    for k, v in Codebook_Temas.items():
        temas_contexto[k] = v.get("descricao", "Sem descrição definida até o momento.")

    prompt = (
        f"CONTEXTO GLOBAL - TEMAS EXISTENTES E SEUS ESCOPOS: {json.dumps(temas_contexto, ensure_ascii=False)}\n\n"
        "Abaixo está a lista de novos subtemas órfãos. Para cada subtema, é fornecida sua 'definicao_teorica' e uma amostra de 'citacoes_empiricas' basais.\n\n"
        "AÇÃO EXIGIDA:\n"
        "1. Utilize a definição e as citações para auditar a abrangência real do subtema e elevá-lo ao nível macro de abstração (Temas).\n\n"
        "DIRETRIZ DE INTEGRALIDADE TAXONÔMICA E AUTONOMIA (REPLICABILIDADE CIENTÍFICA):\n"
        "- Analise o subtema órfão em relação aos TEMAS EXISTENTES. Se o conceito se alinhar, complementar ou expandir o núcleo de significado de um tema já criado, REUTILIZE o nome exato da chave correspondente.\n"
        "- Caso o subtema órfão apresente um fenômeno categoricamente inédito, independente e com alto poder de resposta ao Objetivo da Pesquisa, crie um novo Tema Estruturante.\n"
        "- ATENÇÃO: É permitido que um Tema seja composto por apenas um único subtema, desde que sua relevância científica e autonomia conceitual justifiquem essa independência. Não force agrupamentos artificiais com categorias semanticamente distantes apenas para gerar simetria horizontal.\n"
        "- É EXPRESSAMENTE PROIBIÇÃO criar um novo Tema Estruturante se o conceito puder ser acomodado ou absorvido pelo escopo de um dos TEMAS EXISTENTES informados acima.\n"
        "- Evite pulverizar a análise: nomes de temas devem ser abstratos, incisivos e contar uma história conceitual unificada sobre os dados (MÁXIMO de 10 temas no projeto consolidado).\n"
        "- Se houver similaridade, REUTILIZE o nome exato da chave do Tema existente.\n"
        "- O limite máximo absoluto permanece em 10 temas globais para todo o projeto corporativo, forçando a síntese apenas quando houver convergência real de significado.\n"
        "- Para cada alocação, preencha a 'justificativa_alocacao' demonstrando o nexo causal. Caso crie um tema INÉDITO, forneça sua 'definicao_operacional_curta' de controle (1-2 frases).\n\n"
        f"SUBTEMAS ÓRFÃOS PARA ANÁLISE:\n{json.dumps(subtemas_orfaos, ensure_ascii=False)}\n\n"
        "Retorne o mapeamento relacional estritamente conforme o schema JSON solicitado."
    )
                
    ctx_log = f"FASE 3 | Mapeamento de {len(subtemas_orfaos)} subtemas órfãos"
    resultado = invocar_llm(prompt, list[MapeamentoTema], instrucao, log_contexto=ctx_log)
    
    for item in resultado:
        subtema_nome = item["subtema"].upper().strip()
        tema_nome = item["tema_alocado"].upper().strip()
        
        if tema_nome not in Codebook_Temas:
            Codebook_Temas[tema_nome] = {
                "descricao": "", 
                "definicao_operacional_curta": item.get("definicao_operacional_curta", "").strip(),
                "subtemas_vinculados": [], 
                "justificativas_agrupamento": [],
                "precisa_revisao": True
            }
            
        if subtema_nome not in Codebook_Temas[tema_nome]["subtemas_vinculados"]:
            Codebook_Temas[tema_nome]["subtemas_vinculados"].append(subtema_nome)
            Codebook_Temas[tema_nome]["justificativas_agrupamento"].append({
                "subtema": subtema_nome,
                "justificativa": item.get("justificativa_alocacao", "")
            })
            Codebook_Temas[tema_nome]["precisa_revisao"] = True
            
        print(f"  [MAP] {subtema_nome} -> {tema_nome}")

def executar_fase_intermediaria_3():
    instrucao = (
        "Você é um Analista Qualitativo Sênior. "
        "Suas atividades está centrada em um foco o foco máximo que é o de responder ou identificar elementos que nos ajude a analisar o objetivo da pesquisa. Pois, como componente de uma equipe de pesquisa científica deve colaborar para que suas conclusões sejam imparciais porém focados em atingir o obejtivo de pesquisa. Isto é, subtemas e temas que estejam distântes do Objetivo de Pesquisa deve ser ignorado.\n"
        "Objetivo da Pesquisa: Como Automação Inteligente pode ser efetivamente implementada para aumentar as capacidades cognitivas e contribuindo para o processo decisório e qual framework emerge desse processo de implementação.\n\n"
        "DEFINIÇÕES CONCEITUAIS (BRAUN & CLARKE):\n"
        "- CÓDIGO: Nível granular e fundamental. Ferramenta analítica básica que captura uma única observação ou faceta da informação (obviedade ou significado latente). Não é um conceito amplo, mas uma etiqueta específica para um trecho de dados.\n"
        "- SUBTEMA: Nível estrutural subordinado. Aplicado estritamente para organizar temas amplos ou complexos. Demonstra a hierarquia da informação, estruturando nuances, dimensões ou categorias secundárias do conceito central pai.\n"
        "- TEMA: Alto nível de abstração. Padrão de significado compartilhado em todo o conjunto de dados, unido por um conceito central organizador. Atua como um cristal multifacetado que reúne diversos códigos para contar uma história interpretativa sobre os dados em relação à pergunta de pesquisa. Proibido criar temas que sejam meros resumos de tópicos ou categorias descritivas de uma palavra (ex: 'Vantagens' ou 'Barreiras').\n\n"
        "DIRETRIZES PARA REDAÇÃO DAS DEFINIÇÕES:\n"
        "Para cada elemento gerado (código, subtema ou tema), forneça uma definição rigorosa e analítica formatada como uma descrição narrativa concisa (2 a 4 frases). A definição deve obrigatoriamente:\n"
        "1. Essência: Explicar a ideia central e o conteúdo do padrão identificado.\n"
        "2. Escopo: Demarcar claramente as fronteiras do elemento, especificando o que ele abrange e o que não abrange, para eliminar sobreposição com outros elementos.\n"
        "3. Relevância: Explicar a importância conceitual do elemento, indicando por que ele responde à pergunta de pesquisa e como se conecta à narrativa geral dos dados.\n\n"
        "Sua tarefa é gerar uma Descrição Mestre de alto nível científico (Q1) para um Tema Estruturante, baseando-se nas evidências basais agregadas de todos os subtemas que o compõem.\n\n"
        "Preserve expressões idiomáticas, gírias ou termos técnicos específicos usados pelo entrevistado. Mantenha erros gramaticais ou expressões coloquiais originais para capturar integralmente o contexto cultural, social e as nuances de significado latente."
    )

    for tema, dados in Codebook_Temas.items():
        if dados.get("descricao") and not dados.get("precisa_revisao"):
            continue

        # RESTAURAÇÃO DO MOTOR DE CAPTURA: Coleta exaustivamente as citações das categorias filhas
        evidencias = []
        for sub_nome in dados.get("subtemas_vinculados", []):
            if sub_nome in Codebook_Subtemas:
                for cod_nome in Codebook_Subtemas[sub_nome].get("codigos_vinculados", []):
                    if cod_nome in Codebook_Codigos:
                        evidencias.extend(Codebook_Codigos[cod_nome].get("evidencias_brutas", []))

        evs_unicas = [dict(t) for t in {tuple(d.items()) for d in evidencias}]
        
        # EXAUSTÃO COMPUTAÇÃO: Processamento de 100% dos dados para síntese final de alto impacto (Q1)
        amostra = evs_unicas

        descricao_anterior = dados.get("descricao", "")
        def_inicial = dados.get("definicao_operacional_curta", "Não registrada.")
        justificativas_historicas = dados.get("justificativas_agrupamento", [])
        prompt = (
            f"NOME DO TEMA ESTRUTURANTE: {tema}\n"
            f"DEFINIÇÃO OPERACIONAL INICIAL: {def_inicial}\n"
            f"JUSTIFICATIVAS DE ACOPLAMENTO DOS SUBTEMAS FILHOS: {json.dumps(justificativas_historicas, ensure_ascii=False)}\n"
            f"DESCRIÇÃO CONCEITUAL ACUMULADA DO TEMA: {descricao_anterior if descricao_anterior else 'Inexistente. Construção inicial do macro-padrão.'}\n"
            f"EVIDÊNCIAS EMPÍRICAS CONSOLIDADAS (Insumos dos Códigos e Subtemas): {json.dumps(amostra, ensure_ascii=False)}\n\n"
            "AÇÃO EXIGIDA:\n"
            "Com base no pacote epistêmico completo fornecido, gere ou refine a Descrição Mestre teórica definitiva deste Tema Estruturante.\n"
            "Aplique rigorosamente as DIRETRIZES PARA REDAÇÃO DAS DEFINIÇÕES presentes na instrução de sistema (Essência, Escopo e Relevância).\n\n"
            "DIRETRIZ DE HIPER-FAGOCITOSE EPISTÊMICA E SANEAMENTO DE MACRO-TEMAS:\n"
            "- Esta é a camada de maior nível de abstração da pesquisa. Avalie se o acúmulo de subtemas gerou redundâncias ou discussões que deveriam ser fundidas sob uma única narrativa unificada.\n"
            "- Force a centralização teórica absoluta: use a descrição mestre para consolidar o tema de modo que ele absorva variações de categorias adjacentes próximas, eliminando qualquer flutuação horizontal.\n"
            "- Trace o limite conceitual do construto usando o critério de 'Escopo', deixando claro e explícito para a banca examinadora o que este Tema responde e o que ele categoricamente NÃO abrange.\n\n"
            "REGRA LINGUÍSTICA ABSOLUTA: Baseie-se fortemente nas falas reais para manter o dialeto, termos corporativos, jargões e erros gramaticais originais do entrevistado, capturando o real significado latente do fenômeno corporativo investigado.\n\n"
            "Retorne a síntese formatada estritamente conforme o schema JSON de descrição."
        )
    
        ctx_log = f"SÍNTESE INT. 3 | Tema: {tema}"
        resultado = invocar_llm(prompt, DescricaoMestre, instrucao, log_contexto=ctx_log)
        Codebook_Temas[tema]["descricao"] = resultado["descricao"]
        Codebook_Temas[tema]["precisa_revisao"] = False
        
        # INTERCEPTAÇÃO DE FAGOCITOSE: Alinha a Matriz de Saturação no nível macro de abstração
        lista_mortas = resultado.get("elementos_fagocitados", [])
        registrar_e_executar_fagocitose("TEMA", tema, lista_mortas)
        
        print(f"  [SÍNTESE TEMA] {tema} -> Atualizado/Escrito.")

def registrar_e_executar_fagocitose(tipo_fase: str, chave_destinataria: str, chaves_fagocitadas: list[str]):
    """
    Sincroniza a memória física dos dicionários e o arquivo de auditoria 
    sempre que a LLM decreta uma fusão de categorias em tempo de execução.
    """
    if not chaves_fagocitadas:
        return
        
    historico = {}
    if os.path.exists(ARQUIVO_HISTORICO_FUSOES):
        with open(ARQUIVO_HISTORICO_FUSOES, 'r', encoding='utf-8') as f:
            historico = json.load(f)
            
    for chave_morta in chaves_fagocitadas:
        chave_morta_clean = chave_morta.upper().strip()
        if chave_morta_clean == chave_destinataria or not chave_morta_clean:
            continue
            
        # Executa a fusão física na memória do pipeline
        if tipo_fase == "SUBTEMA" and chave_morta_clean in Codebook_Subtemas:
            # Transfere os códigos vinculados para o subtema sobrevivente
            for cod in Codebook_Subtemas[chave_morta_clean].get("codigos_vinculados", []):
                if cod not in Codebook_Subtemas[chave_destinataria]["codigos_vinculados"]:
                    Codebook_Subtemas[chave_destinataria]["codigos_vinculados"].append(cod)
            # Remove a chave duplicada para zerar o ganho falso na Matriz de Saturação
            del Codebook_Subtemas[chave_morta_clean]
            
        elif tipo_fase == "TEMA" and chave_morta_clean in Codebook_Temas:
            # Transfere os subtemas vinculados para o tema sobrevivente
            for sub in Codebook_Temas[chave_morta_clean].get("subtemas_vinculados", []):
                if sub not in Codebook_Temas[chave_destinataria]["subtemas_vinculados"]:
                    Codebook_Temas[chave_destinataria]["subtemas_vinculados"].append(sub)
            del Codebook_Temas[chave_morta_clean]
            
        # Registra a mutação no arquivo JSON de auditoria histórica
        log_id = f"{tipo_fase}_{time.strftime('%Y%m%d_%H%M%S')}"
        historico[log_id] = {
            "Tipo": tipo_fase,
            "Chave_Absorvida": chave_morta_clean,
            "Chave_Destino_Consolidada": chave_destinataria,
            "Data_Hora": time.strftime('%Y-%m-%d %H:%M:%S')
        }
        print(f"  [FAGOCITOSE OPERADA] {chave_morta_clean} foi totalmente colapsada em -> {chave_destinataria}")
        
    with open(ARQUIVO_HISTORICO_FUSOES, 'w', encoding='utf-8') as f:
        json.dump(historico, f, ensure_ascii=False, indent=4)

In [6]:
# 6. ENGENHARIA DE RELATÓRIOS, MATRIZES E VISUALIZAÇÃO PANORÂMICA
# ==============================================================================
# DESCRIÇÃO ARQUITETURAL:
# Responsável pela materialização dos dados em disco para consumo humano.
# 1. Exportação Relacional (Excel): Constrói a rastreabilidade Bottom-Up 
#    (Código -> Subtema -> Tema), consolida o Codebook com as Descrições Mestres 
#    e gera a Matriz de Saturação Teórica.
# 2. Visualização Científica (Grafos): Implementa um algoritmo de renderização 
#    de Grafo Direcionado (DiGraph) para gerar um Dendrograma Hierárquico. 
#    Utiliza cálculo geométrico para alinhamento Top-Down (Esquerda para Direita) 
#    e escala de canal Alpha (transparência) para representar profundidade analítica.
# ==============================================================================

def exportar_artefatos(sufixo_arquivo=""):
    """
    Gera a documentação relacional do estudo em formato de planilha Excel.
    
    Lógica de Negócio:
      1. Extrai a base de dados brutos (Rastreabilidade_Base).
      2. Mapeia a hierarquia de cada código alocado em tempo real.
      3. Constrói o Codebook final concatenando construtos teóricos gerados pelas LLMs.
      4. Acopla a matriz matemática de saturação para validação empírica.
      
    Args:
        sufixo_arquivo (str): Identificador dinâmico anexado ao nome do arquivo exportado.
    """
    df_rastreabilidade = pd.DataFrame(Rastreabilidade_Base)
    if df_rastreabilidade.empty:
        return

    mapa_cod_sub = {cod: sub for sub, dados in Codebook_Subtemas.items() for cod in dados.get("codigos_vinculados", [])}
    mapa_sub_tema = {sub: tema for tema, dados in Codebook_Temas.items() for sub in dados.get("subtemas_vinculados", [])}

    df_rastreabilidade['Subtema_Atribuido'] = df_rastreabilidade.get('Codigo_Atribuido', pd.Series(dtype=str)).map(mapa_cod_sub)
    df_rastreabilidade['Tema_Atribuido'] = df_rastreabilidade.get('Subtema_Atribuido', pd.Series(dtype=str)).map(mapa_sub_tema)
    
    linhas_codebook = []
    for tema, d_tema in Codebook_Temas.items():
        desc_tema = d_tema.get("descricao", "")
        for sub in d_tema.get("subtemas_vinculados", []):
            desc_sub = Codebook_Subtemas.get(sub, {}).get("descricao", "")
            for cod in Codebook_Subtemas.get(sub, {}).get("codigos_vinculados", []):
                evs = Codebook_Codigos.get(cod, {}).get("evidencias_brutas", [])
                textos_amostra = "\n--- \n".join([f'"{e["citacao"]}" ({e["justificativa"]})' for e in evs[:3]])
                
                linhas_codebook.append({
                    "Tema": tema, 
                    "Descricao_Tema": desc_tema,
                    "Subtema": sub, 
                    "Descricao_Subtema": desc_sub,
                    "Codigo": cod, 
                    "Descricao_Codigo": Codebook_Codigos.get(cod, {}).get("descricao", ""),
                    "Exemplos_Citacoes": textos_amostra
                })
    
    df_codebook = pd.DataFrame(linhas_codebook)
    df_saturacao = gerar_matriz_saturacao_ampliada()
    
    nome_arquivo = f"Resultado_Analise_Tematica_{sufixo_arquivo}.xlsx"
    caminho_saida = os.path.join(DIRETORIO_ENTREVISTAS, nome_arquivo)
    
    with pd.ExcelWriter(caminho_saida, engine='openpyxl') as writer:
        df_rastreabilidade.to_excel(writer, sheet_name="Dados_Codificados", index=False)
        df_codebook.to_excel(writer, sheet_name="Codebook_Consolidado", index=False)
        if not df_saturacao.empty: 
            df_saturacao.to_excel(writer, sheet_name="Matriz_Saturacao", index=False)

def gerar_matriz_saturacao_ampliada() -> pd.DataFrame:
    """
    Rastreia o ponto de saturação teórica do estudo.
    
    Lógica de Negócio:
      Itera cronologicamente sobre os arquivos e calcula a taxa de surgimento 
      de novos elementos (Códigos, Subtemas e Temas). A interrupção no surgimento 
      de novos Códigos indica que a amostra atingiu exaustão representativa empírica.
      
    Returns:
        pd.DataFrame: Tabela contendo a evolução incremental da volumetria estrutural.
    """
    if not Rastreabilidade_Base: 
        return pd.DataFrame()
        
    mapa_cod_sub = {cod: sub for sub, dados in Codebook_Subtemas.items() for cod in dados.get("codigos_vinculados", [])}
    mapa_sub_tema = {sub: tema for tema, dados in Codebook_Temas.items() for sub in dados.get("subtemas_vinculados", [])}

    # Preservação da ordem cronológica de processamento
    arquivos_ordem = []
    for item in Rastreabilidade_Base:
        if item["Arquivo_Origem"] not in arquivos_ordem: 
            arquivos_ordem.append(item["Arquivo_Origem"])

    codigos_vistos = set()
    subtemas_vistos = set()
    temas_vistos = set()
    matriz_ampliada = []

    for arquivo in arquivos_ordem:
        cods_neste_arquivo = {item["Codigo_Atribuido"] for item in Rastreabilidade_Base if item["Arquivo_Origem"] == arquivo}
        novos_cods = novos_subs = novos_temas = 0

        # Contagem diferencial utilizando Teoria de Conjuntos (Set)
        for cod in cods_neste_arquivo:
            if cod not in codigos_vistos: 
                novos_cods += 1
                codigos_vistos.add(cod)
            
            sub = mapa_cod_sub.get(cod)
            if sub and sub not in subtemas_vistos: 
                novos_subs += 1
                subtemas_vistos.add(sub)
            
            tema = mapa_sub_tema.get(sub) if sub else None
            if tema and tema not in temas_vistos: 
                novos_temas += 1
                temas_vistos.add(tema)

        matriz_ampliada.append({
            "Nome_Arquivo": arquivo,
            "Novos_Codigos": novos_cods, "Total_Codigos": len(codigos_vistos),
            "Novos_Subtemas": novos_subs, "Total_Subtemas": len(subtemas_vistos),
            "Novos_Temas": novos_temas, "Total_Temas": len(temas_vistos)
        })
        
    return pd.DataFrame(matriz_ampliada)


def gerar_graficos_cientificos(sufixo_iteracao="Final"):
    """
    Gera a representação visual macro do modelo teórico resultante.
    
    Lógica de Negócio e Algoritmo Gráfico:
      1. Instancia um Grafo Direcionado (nx.DiGraph).
      2. Mapeia posições (X, Y) programaticamente. 
         - X=0 (Temas), X=1 (Subtemas), X=2 (Códigos).
         - Y é decrementado linearmente pelas 'folhas' (Códigos) para evitar sobreposição.
      3. Nós superiores (Subtemas e Temas) têm seu eixo Y calculado como o 
         ponto médio (média aritmética) do eixo Y de seus respectivos nós filhos.
      4. Aplica codificação visual baseada em heurísticas de percepção:
         - Matiz (Color): Diferencia os Temas macroestruturantes (Palette tab20).
         - Saturação (Alpha): Diferencia o nível de abstração (Tema=95%, Subtema=55%, Código=25%).
        Parâmetros:
        - sufixo_iteracao (str): Identificador numérico anexado ao nome do arquivo de saída.
            Garante a preservação histórica do estado da árvore categórica para trilha de auditoria.
    """
    print("\n--- INICIANDO GERAÇÃO DE MAPA TEMÁTICO PANORÂMICO (ÁRVORE HIERÁRQUICA) ---")
    
    G = nx.DiGraph()
    pos = {}
    node_colors = {}
    node_alphas = {}
    node_levels = {} # [NOVO] Dicionário para rastrear a hierarquia do nó
    
    # Definição de paleta para diferenciação categórica horizontal
    cmap = plt.get_cmap("tab20")
    temas = list(Codebook_Temas.keys())
    
    y_atual = 0
    largura_quebra = 40 # Limite de caracteres para quebra de linha nas caixas do grafo
    
    # =====================================================================
    # MOTOR DE CÁLCULO DE COORDENADAS ESPACIAIS (GRID DISTRIBUÍDO)
    # =====================================================================
    largura_quebra_tema = 30
    largura_quebra_codigo = 30 # Mais estreito para caber lado a lado
    max_colunas_codigos = 4    # Quantidade de códigos na horizontal
    passo_x_codigo = 0.8       # Distância horizontal entre códigos
    passo_y_codigo = 0.8       # Distância vertical entre linhas do grid

    for i, tema in enumerate(temas):
        cor_base = cmap(i % 20)
        tema_lbl = "\n".join(textwrap.wrap(tema, largura_quebra_tema))
        
        if tema_lbl not in pos:
            G.add_node(tema_lbl)
            node_colors[tema_lbl] = cor_base
            node_alphas[tema_lbl] = 0.95
            node_levels[tema_lbl] = "tema"
        
        subtemas = Codebook_Temas[tema].get("subtemas_vinculados", [])
        y_tema_start = y_atual
        
        for sub in subtemas:
            sub_lbl = "\n".join(textwrap.wrap(sub, largura_quebra_tema))
            
            if sub_lbl not in pos:
                G.add_node(sub_lbl)
                node_colors[sub_lbl] = cor_base
                node_alphas[sub_lbl] = 0.55
                node_levels[sub_lbl] = "subtema"
            
            G.add_edge(tema_lbl, sub_lbl)
            
            codigos = Codebook_Subtemas.get(sub, {}).get("codigos_vinculados", [])
            has_new_codes = False
            y_min_neste_subtema = y_atual
            
            # Distribuição em Matriz Bidimensional (Grid)
            for idx, cod in enumerate(codigos):
                cod_lbl = "\n".join(textwrap.wrap(cod, largura_quebra_codigo))
                
                if cod_lbl not in pos:
                    G.add_node(cod_lbl)
                    node_colors[cod_lbl] = cor_base
                    node_alphas[cod_lbl] = 0.55  # Alterado de 0.25 para 0.55 (mesma cor do subtema)
                    node_levels[cod_lbl] = "codigo"
                    
                    # Cálculo de posição na matriz
                    coluna = idx % max_colunas_codigos
                    linha = idx // max_colunas_codigos
                    
                    pos_x = 2.5 + (coluna * passo_x_codigo)
                    pos_y = y_atual - (linha * passo_y_codigo)
                    
                    pos[cod_lbl] = (pos_x, pos_y)
                    
                    if pos_y < y_min_neste_subtema:
                        y_min_neste_subtema = pos_y
                        
                    has_new_codes = True
                    
                G.add_edge(sub_lbl, cod_lbl)
            
            # Centralização do Subtema e avanço do ponteiro Y
            if has_new_codes:
                pos[sub_lbl] = (1.0, (y_atual + y_min_neste_subtema) / 2.0)
                y_atual = y_min_neste_subtema - 1.2 # Margem de respiro para o próximo subtema
            elif sub_lbl not in pos: 
                pos[sub_lbl] = (1.0, y_atual)
                y_atual -= 1.2
        
        # Centralização Geométrica do Tema
        if y_tema_start != y_atual:
            # Compensa a última margem de respiro inserida no loop de subtemas
            pos[tema_lbl] = (0, (y_tema_start + y_atual + 1.2) / 2.0) 
        elif tema_lbl not in pos:
            pos[tema_lbl] = (0, y_atual)
            y_atual -= 1.5

    if not G.nodes():
        print("  [AVISO] Dados insuficientes para gerar o mapa panorâmico.")
        return

    # =====================================================================
    # MOTOR DE RENDERIZAÇÃO MATPLOTLIB
    # =====================================================================
    altura_canvas = max(12.0, abs(y_atual) * 0.9)
    largura_canvas = 40.0 
    plt.figure(figsize=(largura_canvas, altura_canvas))
    
    # zorder=1: Força as linhas para a camada de fundo
    nx.draw_networkx_edges(G, pos, edge_color="#CCCCCC", arrows=True, arrowstyle="-|>", arrowsize=10, width=1.2)
    
    for node in G.nodes():
        x, y = pos[node]
        cor = node_colors.get(node, "#DDDDDD")
        alpha = node_alphas.get(node, 0.5)
        nivel = node_levels.get(node, "codigo") # Recupera a classificação estrutural
        
        rgba_color = mcolors.to_rgba(cor, alpha=alpha)
        
        # Lógica de Controle Tipográfico por Nível
        if nivel == "tema":
            tamanho_fonte = 16
            peso_fonte = 'black' # Extra-bold para máximo destaque
        elif nivel == "subtema":
            tamanho_fonte = 13
            peso_fonte = 'bold'
        else:
            tamanho_fonte = 10
            peso_fonte = 'bold'
        
        plt.text(x, y, node, 
                 fontsize=tamanho_fonte, 
                 fontweight=peso_fonte,
                 fontfamily='sans-serif',
                 ha='center', va='center',
                 zorder=10, 
                 bbox=dict(boxstyle="round,pad=0.6", facecolor=rgba_color, edgecolor="#555555", linewidth=1.2))

    plt.xlim(-0.5, 3.0 + (max_colunas_codigos * passo_x_codigo))
    plt.ylim(y_atual - 1, 1.5) 
    plt.axis("off")
    
    # Montagem da legenda estrutural de profundidade
    legendas = [
        mpatches.Patch(facecolor='gray', alpha=0.95, edgecolor='black', label='Tema (Nível 1)'),
        mpatches.Patch(facecolor='gray', alpha=0.55, edgecolor='black', label='Subtema (Nível 2)'),
        mpatches.Patch(facecolor='gray', alpha=0.25, edgecolor='black', label='Código (Nível 3)')
    ]
    plt.legend(handles=legendas, loc='upper center', ncol=3, bbox_to_anchor=(0.5, 1.02), 
               title="Árvore Hierárquica Relacional (Cada matiz de cor representa 1 Tema Macroestruturante)", 
               title_fontsize=11)
    
    nome_arquivo = f"Mapa_Tematico_Panoramico_{sufixo_iteracao}.png"
    caminho_saida = os.path.join(DIRETORIO_GRAFICOS, nome_arquivo)
    plt.savefig(caminho_saida, format="png", dpi=300, bbox_inches="tight")
    plt.close()
    
    print(f"  [EXPORTAÇÃO GRÁFICA] Mapa Panorâmico exportado em: {caminho_saida}")

def gerar_grafico_frequencia_caixas():
    """
    Gera representação Treemap.
    Objetivo: Mapear a densidade empírica onde a área geométrica da caixa 
    é estritamente proporcional ao volume de incidência (repetições) do código.
    """
    print("\n--- INICIANDO GERAÇÃO DE MAPA DE CAIXAS (TREEMAP) ---")
    
    labels = []
    sizes = []
    colors = []
    cmap = plt.get_cmap("tab20")
    
    for i, (tema, d_tema) in enumerate(Codebook_Temas.items()):
        cor_base = cmap(i % 20)
        for sub in d_tema.get("subtemas_vinculados", []):
            for cod in Codebook_Subtemas.get(sub, {}).get("codigos_vinculados", []):
                evidencias = Codebook_Codigos.get(cod, {}).get("evidencias_brutas", [])
                frequencia = len(evidencias)
                if frequencia > 0:
                    rotulo = f"{cod}\n(n={frequencia})"
                    labels.append(rotulo)
                    sizes.append(frequencia)
                    colors.append(mcolors.to_rgba(cor_base, alpha=0.6))
    
    if not sizes:
        print("  [AVISO] Dados insuficientes para o Treemap.")
        return

    plt.figure(figsize=(16, 10))
    squarify.plot(sizes=sizes, label=labels, color=colors, alpha=0.8,
                  text_kwargs={'fontsize': 10, 'fontweight': 'bold', 'wrap': True})
    
    plt.axis('off')
    plt.title("Densidade Temática: Proporção de Códigos por Saturação de Evidências", fontsize=14, fontweight='bold', pad=20)
    
    caminho_saida = os.path.join(DIRETORIO_GRAFICOS, "Mapa_Frequencia_Caixas.png")
    plt.savefig(caminho_saida, format="png", dpi=300, bbox_inches="tight")
    plt.close()
    
    print(f"  [EXPORTAÇÃO GRÁFICA] Treemap exportado em: {caminho_saida}")

def gerar_grafico_hierarquico_proporcional():
    """
    Gera representação hierárquica proporcional usando go.Icicle.
    A paleta de cores foi antecipada para permitir a estilização inline dos blocos de códigos.
    """
    print("\n--- INICIANDO GERAÇÃO DE MAPA HIERÁRQUICO (MODO TABELA PROPORCIONAL) ---")
    
    freq_por_tema = {}
    freq_por_subtema = {}
    codigos_agrupados = {}
    
    # 1. Definição antecipada do mapeamento de cores para alimentar as strings HTML
    cmap = px.colors.qualitative.Pastel
    temas_unicos = list(Codebook_Temas.keys())
    color_map = {tema: cmap[i % len(cmap)] for i, tema in enumerate(temas_unicos)}
    
    for tema, d_tema in Codebook_Temas.items():
        cor_tema = color_map.get(tema, "#CCCCCC")
        soma_tema = 0
        for sub in d_tema.get("subtemas_vinculados", []):
            soma_sub = 0
            lista_codigos = []
            for cod in Codebook_Subtemas.get(sub, {}).get("codigos_vinculados", []):
                evidencias = Codebook_Codigos.get(cod, {}).get("evidencias_brutas", [])
                freq = len(evidencias)
                if freq > 0:
                    soma_sub += freq
                    # Injeção de estilo inline para emular caixas coloridas nos códigos individuais
                    lista_codigos.append(
                        f"<span style='background-color: {cor_tema}; padding: 4px 8px; "
                        f"border: 1px solid #555555; border-radius: 4px; display: inline-block; "
                        f"margin: 2px;'>{cod} (n={freq})</span>"
                    )
            
            if soma_sub > 0:
                freq_por_subtema[f"T_{tema}_S_{sub}"] = soma_sub
                
                chunk_size = 2 
                linhas = []
                for i in range(0, len(lista_codigos), chunk_size):
                    linhas.append(" &nbsp;&nbsp; ".join(lista_codigos[i:i+chunk_size]))
                
                codigos_agrupados[f"T_{tema}_S_{sub}_C"] = "<br><br>".join(linhas)
                soma_tema += soma_sub
                
        if soma_tema > 0:
            freq_por_tema[f"T_{tema}"] = soma_tema

    if not freq_por_tema:
        print("  [AVISO] Dados insuficientes para o gráfico hierárquico.")
        return

    ids = []
    labels = []
    parents = []
    values = []

# Construção: Nível 1 - Temas
    for tema_id, freq in freq_por_tema.items():
        ids.append(tema_id)
        # BURLA DE LIMITE HORIZONTAL: Empilhamento vertical de palavras
        # nome_tema = tema_id.replace("T_", "").upper()
        nome_tema = tema_id[2:].upper()
        labels.append(f"<b>{nome_tema}</b><br><br>(n={freq})")
        parents.append("") 
        values.append(freq)

    # Construção: Nível 2 - Subtemas
    for sub_id, freq in freq_por_subtema.items():
        ids.append(sub_id)
        # BURLA DE LIMITE HORIZONTAL: Empilhamento vertical de palavras
        nome_sub = sub_id.split("_S_")[1].title().replace(" ", "<br>") 
        labels.append(f"<b>{nome_sub}</b><br><br>(n={freq})")
        tema_parent = sub_id.split("_S_")[0]
        parents.append(tema_parent)
        values.append(freq)

    # Construção: Nível 3 - Códigos (Formatação HTML preservada)
    for cod_id, texto_html in codigos_agrupados.items():
        ids.append(cod_id)
        labels.append(texto_html)
        # sub_parent = cod_id.replace("_C", "")
        sub_parent = cod_id[:-2]

        parents.append(sub_parent)
        values.append(freq_por_subtema[sub_parent])

    # Força o contentor de fundo da última coluna a ser transparente
    colors = []
    for node_id in ids:
        if node_id.endswith("_C"):
            colors.append("rgba(0,0,0,0)") 
        else:
            tema_base = node_id.split("_S_")[0]
            # colors.append(color_map.get(tema_base.replace("T_", ""), "#CCCCCC"))
            colors.append(color_map.get(tema_base[2:], "#CCCCCC"))

    fig = go.Figure(go.Icicle(
        ids=ids,
        labels=labels,
        parents=parents,
        values=values,
        branchvalues="total",
        marker=dict(colors=colors, line=dict(color='#FFFFFF', width=1.5)),
        textinfo="label",
        # Reduzido de 30 para 14. Um valor menor impede que o algoritmo do Plotly 
        # entre em "pânico" tentando espremer textos gigantes, resultando em uma 
        # renderização nativa mais nítida e equilibrada.
        textfont=dict(size=14, color="#111111"), 
        pathbar=dict(visible=False),
        tiling=dict(pad=2) 
    ))

    annotations = [
        dict(x=0.16, y=1.03, xref='paper', yref='paper', text="<b>TEMA</b>", showarrow=False, font=dict(size=14), xanchor="center"),
        dict(x=0.50, y=1.03, xref='paper', yref='paper', text="<b>SUBTEMA</b>", showarrow=False, font=dict(size=14), xanchor="center"),
        dict(x=0.83, y=1.03, xref='paper', yref='paper', text="<b>CÓDIGOS E EVIDÊNCIAS</b>", showarrow=False, font=dict(size=14), xanchor="center")
    ]

    fig.update_layout(
        margin=dict(t=50, l=10, r=10, b=10),
        annotations=annotations,
        # REDUZIDO: A largura total foi diminuída (de 1400 para 1200) para forçar as colunas a ficarem mais próximas. 
        # Se os textos dos códigos começarem a quebrar de forma indesejada, aumente para 1250 ou 1300.
        width=1200, 
        height=900
    )

    caminho_saida_png = os.path.join(DIRETORIO_GRAFICOS, "Mapa_Hierarquico_Proporcional.png")
    fig.write_image(caminho_saida_png, scale=2)
    
    print(f"  [EXPORTAÇÃO GRÁFICA] Gráfico de áreas exportado em: {caminho_saida_png}")

In [7]:
# 7. Orquestrador: Novo fluxo de controle incremental. Lê as fontes da verdade, processa as 6
# etapas apenas para o arquivo da iteração e grava as modificações. 
# ==============================================================================

def orquestrar_pipeline_incremental():
    # --- NOVO BLOCO: MOTOR DE CAPTURA INCREMENTAL DO CONSOLE ---
    import sys
    class LoggerDuplo(object):
        def __init__(self, caminho_arquivo):
            self.terminal = sys.stdout
            # 'a' garante o modo append: se o código parar e voltar, ele continua de onde parou
            self.log = open(caminho_arquivo, "a", encoding="utf-8")
            
        def write(self, mensagem):
            self.terminal.write(mensagem)
            self.log.write(mensagem)
            self.log.flush() # Força a gravação imediata no disco para evitar perda em crashes
            
        def flush(self):
            self.terminal.flush()
            self.log.flush()

    # Ativa o redirecionador de fluxo do sistema para o arquivo mestre de auditoria
    sys.stdout = LoggerDuplo(ARQUIVO_LOG_CONSOLE)
    
    # Registra o marcador de inicialização/retomada com carimbo de tempo real
    print(f"\n[{time.strftime('%Y-%m-%d %H:%M:%S')}] --- INICIALIZAÇÃO DO FLUXO DE AUDITORIA DE CONSOLE ---")
    # -------------------------------------------------------------------------

    arquivos_alvo = ordenar_arquivos_cronologicamente(DIRETORIO_ENTREVISTAS)
    if not arquivos_alvo:
        raise FileNotFoundError(f"Nenhum arquivo válido (.txt) localizado em {DIRETORIO_ENTREVISTAS}.")

    for idx, arquivo in enumerate(arquivos_alvo, start=1):
        # BARREIRA DE ENTRADA (Short-Circuit)
        nome_grafico_esperado = f"Mapa_Tematico_Panoramico_{idx}.png"
        caminho_grafico = os.path.join(DIRETORIO_GRAFICOS, nome_grafico_esperado)
        
        if os.path.exists(caminho_grafico):
            print(f"[{arquivo}] Iteração {idx} detectada como concluída. Processamento ignorado.")
            continue 
            
        caminho_arquivo = os.path.join(DIRETORIO_ENTREVISTAS, arquivo)
        print(f"\n{'='*60}\nINICIANDO PROCESSAMENTO: {arquivo} (Iteração {idx})\n{'='*60}")
        
        carregar_estado_json()
        
        print("\n[ETAPA 1] Codificação Primária...")
        executar_fase_1(arquivo)
        salvar_estado_json() 
        
        print("\n[ETAPA 2] Síntese de Novos Códigos...")
        executar_fase_intermediaria_1()
        salvar_estado_json()
        
        print("\n[ETAPA 3] Agrupamento: Códigos Órfãos -> Subtemas...")
        executar_fase_2()
        salvar_estado_json()
        
        print("\n[ETAPA 4] Síntese de Subtemas Sem Descrição...")
        executar_fase_intermediaria_2()
        salvar_estado_json()
        
        print("\n[ETAPA 5] Agrupamento: Subtemas Órfãos -> Temas...")
        executar_fase_3()
        salvar_estado_json()
        
        print("\n[ETAPA 6] Síntese de Temas Sem Descrição...")
        executar_fase_intermediaria_3()
        salvar_estado_json()
        
        exportar_artefatos(arquivo.replace(".txt", ""))
        
        print(f"  [AUDITORIA VISUAL] Renderizando retrato estrutural (Iteração {idx})...")
        gerar_graficos_cientificos(sufixo_iteracao=str(idx))

    print("\n--- PROCESSAMENTO INTEGRAL CONCLUÍDO ---")

In [8]:
# 8. EXECUTAR ANÁLISE COM LLM

if __name__ == "__main__":
    orquestrar_pipeline_incremental()


[2026-06-16 23:40:54] --- INICIALIZAÇÃO DO FLUXO DE AUDITORIA DE CONSOLE ---

INICIANDO PROCESSAMENTO: Entrevistado A1.txt (Iteração 1)

[ETAPA 1] Codificação Primária...
  [FASE 1] Chunk 1/6 processado. | 5 extrações.
  [FASE 1] Chunk 2/6 processado. | 5 extrações.
  [FASE 1] Chunk 3/6 processado. | 5 extrações.
  [FASE 1] Chunk 4/6 processado. | 5 extrações.
  [FASE 1] Chunk 5/6 processado. | 5 extrações.
  [FASE 1] Chunk 6/6 processado. | 5 extrações.

[ETAPA 2] Síntese de Novos Códigos...
  [SÍNTESE CÓDIGO] FRAGMENTAÇÃO DA FONTE DE VERDADE -> Escrito.
  [SÍNTESE CÓDIGO] DISSONÂNCIA COGNITIVA OPERACIONAL -> Escrito.
  [SÍNTESE CÓDIGO] FEEDBACK COMO SUBSÍDIO ESTRATÉGICO -> Escrito.
  [SÍNTESE CÓDIGO] SIMPLIFICAÇÃO COMO AUMENTO COGNITIVO -> Escrito.
  [SÍNTESE CÓDIGO] CENTRALIZAÇÃO DA GOVERNANÇA DA INFORMAÇÃO -> Escrito.
  [SÍNTESE CÓDIGO] ARQUITETURA DA INFORMAÇÃO DIRECIONADA -> Escrito.
  [SÍNTESE CÓDIGO] FRICÇÃO DA FERRAMENTA -> Escrito.
  [SÍNTESE CÓDIGO] REDUNDÂNCIA PROCESSUAL C